In [0]:
data = [
    (101,"Rahul",75000),
    (102,"Priya",85000),
    (103,"Amit",65000)
]

df = spark.createDataFrame(
    data,
    ["emp_id","name","salary"]
)


In [0]:
df.write.format("delta") \
.save("/tmp/employees_delta")

In [0]:
delta_df = spark.read.format("delta") \
.load("/tmp/employees_delta")

display(delta_df)

emp_id,name,salary
101,Rahul,75000
102,Priya,85000
103,Amit,65000


In [0]:
df.write.format("delta") \
.saveAsTable("employees")

In [0]:
%sql
select * from employees

emp_id,name,salary
101,Rahul,75000
102,Priya,85000
103,Amit,65000


In [0]:
%sql
CREATE TABLE employees_delta
(
    emp_id INT,
    name STRING,
    salary INT
)
USING DELTA


In [0]:
%sql
INSERT INTO employees_delta
VALUES
(101,'Rahul',75000),
(102,'Priya',85000)

num_affected_rows,num_inserted_rows
2,2


In [0]:
updates = [

(102,"Priya",90000),
(104,"Sneha",70000)

]

updates_df = spark.createDataFrame(
    updates,
    ["emp_id","name","salary"]
)

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(
    spark,
    "/tmp/employees_delta"
)

delta_table.alias("target") \
.merge(
    updates_df.alias("source"),
    "target.emp_id = source.emp_id"
) \
.whenMatchedUpdateAll() \
.whenNotMatchedInsertAll() \
.execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql DESCRIBE HISTORY employees

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-06-17T09:46:49.000Z,148916151530801,azuser7222_mml.local@karthikirisoutlook.onmicrosoft.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(284788352675040),caf346c7-0031-4712-8a1f-dcfcdc3df863,0617-094348-kce8ps8f-v2n,null,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 3, numOutputBytes -> 1308)",null,Databricks-Runtime/18.2.x-photon-scala2.13


In [0]:
df = spark.read.format("delta") \
.option("versionAsOf",0) \
.load("/tmp/employees_delta")
 
display(df)

emp_id,name,salary
101,Rahul,75000
102,Priya,85000
103,Amit,65000


In [0]:
%sql OPTIMIZE employees

path,metrics
abfss://unity-catalog-storage@dbstoragedaqeulmliu5fu.dfs.core.windows.net/7405615139389364/__unitystorage/catalogs/29ad4bb0-3fe8-43a4-a04c-e30c409a4d1f/tables/d1f5ef78-22c2-49e9-942d-9ff79cbf742d,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1781690536483, 1781690537090, 8, 0, null, List(0, 0), null, 3, 3, 0, 0, null, null)"


In [0]:
%sql 
OPTIMIZE employees
ZORDER BY(name)

path,metrics
abfss://unity-catalog-storage@dbstoragedaqeulmliu5fu.dfs.core.windows.net/7405615139389364/__unitystorage/catalogs/29ad4bb0-3fe8-43a4-a04c-e30c409a4d1f/tables/d1f5ef78-22c2-49e9-942d-9ff79cbf742d,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 1308), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1781690943267, 1781690943828, 8, 0, null, List(0, 0), null, 3, 3, 0, 0, null, null)"


In [0]:
%sql VACUUM employees

path
abfss://unity-catalog-storage@dbstoragedaqeulmliu5fu.dfs.core.windows.net/7405615139389364/__unitystorage/catalogs/29ad4bb0-3fe8-43a4-a04c-e30c409a4d1f/tables/d1f5ef78-22c2-49e9-942d-9ff79cbf742d
